# Assignment 3
## Econ 8310 - Business Forecasting

For homework assignment 3, you will work with [Fashion MNIST](https://github.com/zalandoresearch/fashion-mnist), a more fancier data set.

- You must create a custom data loader as described in the first week of neural network lectures [2 points]
    - You will NOT receive credit for this if you use the pytorch prebuilt loader for Fashion MNIST!
- You must create a working and trained neural network using only pytorch [2 points]
- You must store your weights and create an import script so that I can evaluate your model without training it [2 points]

Highest accuracy score gets some extra credit!

Submit your forked repository URL on Canvas! :) I'll be manually grading this assignment.

Some checks you can make on your own:
- Did you manually process the data or use a prebuilt loader (see above)?
- Does your script train a neural network on the assigned data?
- Did your script save your model?
- Do you have separate code to import your model for use after training?

In [163]:
import pandas as pd
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import requests

#for visualizing
import plotly.express as px

#for model building
import torch
import torch.nn as nn
import torch.nn.functional as F

#file download
import os

#parsing library
import xml.etree.ElementTree as ET

#video library
import cv2

ModuleNotFoundError: No module named 'cv2'

In [ ]:
#saving files from downloaded folder into list
#running through list, and then adding the read xml file into a new list
folder = "C:/Users/andre/Downloads/baseballannotated"
data = []

for current, subfolder, files in os.walk(folder):
    for file in files:
        data.append(os.path.join(current, file))

readData = []

for i in data:
    readData.append(open(i).read())

# print(len(readData))
print(readData[0])



In [ ]:
# importing XML files. video found online using xml.tree.ElementTree on importing and parsing
#https://www.youtube.com/watch?v=5SlemSWGD1g

# importing video files. video found online using openCV module
#https://video.search.yahoo.com/video/play;_ylt=AwrFFynW5c5p2cwB3kf7w8QF;_ylu=c2VjA3NyBHNsawN2aWQEZ3BvcwMx?p=importing+videos+and+separating+video+frames+in+python&vid=c9ce7d4e05e775c0f1823d9d27196bfa&turl=https%3A%2F%2Ftse3.mm.bing.net%2Fth%2Fid%2FOVP.AjTplKi-isv8hx3pXAcsrQHgFo%3Fpid%3DApi%26h%3D360%26w%3D480%26c%3D7%26rs%3D1&rurl=https%3A%2F%2Fwww.youtube.com%2Fwatch%3Fv%3DECCHN1j_lis&tit=Extracting+%3Cb%3EVideo%3C%2Fb%3E+%3Cb%3EFrames%3C%2Fb%3E+Using+OpenCV+%3Cb%3Ein%3C%2Fb%3E+%3Cb%3EPython%3C%2Fb%3E+%7C+%3Cb%3EPython%3C%2Fb%3E+Project&c=0&sigr=qIUGT8oAANaB&sigt=0rHo6zFK58vl&sigi=jNYR9_QI3F_K&fr2=p%3As%2Cv%3Av&h=360&w=480&l=244&age=1656509403&fr=mcafee&type=E210US1357G0&tt=b

#baseball class

class CustomBaseball(Dataset):
  def __init__(self, data):
    self.data = data

  def __len__(self):
    return len(self.data)

  def __getitem__(self, idx):      #extract and separate label, frame, boundbox
        frame = data["frame"].values.reshape(1, 28, 28)                          #need to correct
        #specify dtype to align with default dtype used by weight matrices
        boundbox = torch.tensor(item["boundbox"], dtype=torch.float32)
        #label to tensor
        label = torch.tensor(item["label"])

        #return the image and its corresponding label and boundbox
        return image, boundbox, label

# parse and extract labels from xml file

import xml.etree.ElementTree as ET
import os
import pandas as pd

folder = "/content/videos/IMG"
xmlFiles = []
for file in os.listdir(folder):

    filePath = os.path.join(folder, file)
    tree = ET.parse(filePath)
    root = tree.getroot()

    for track in root.findall("track"):  #need a for loop for each id within xml file
        id = track.attrib["id"]
        label = track.attrib["label"]           #baseball label

        for child in track.findall("box"):               #getting label values from binding boxes
            frame = int(child.attrib["frame"])
            outside = int(child.attrib["outside"])
            xtl = float(child.attrib["xtl"])
            ytl = float(child.attrib["ytl"])
            xbr = float(child.attrib["xbr"])
            ybr = float(child.attrib["ybr"])

            for tag in child:
                if tag.text == "true":
                    moving = 1
                else:
                    moving = 0

            # items = [file, id, label, frame, outside, xtl, ytl, xbr, ybr, moving]
            xmlFiles.append({
                "file": os.path.splitext(file)[0],
                "id": id,
                "label": label,
                "frame": frame,
                "outside": outside,
                "boundbox": [xtl, ytl, xbr, ybr]
            })

print("Files parsed.")

# import and separate frames from videos

import cv2
import os

videoFolder = "/content/videos/Movies"
outputFolder = "/content/videos/captures"

# # creating folder (for google collab)
# outputFolder = "videos/Captures"
# os.makedirs(outputFolder, exist_ok=True)

#for loop to go through videos
for videoFile in os.listdir(videoFolder):
  path = os.path.join(videoFolder, videoFile)

  #creating folder for each video
  videoName = os.path.splitext(videoFile)[0]
  outFolder = os.path.join(outputFolder, videoName)
  os.makedirs(outFolder, exist_ok=True)

  #capture video
  capture = cv2.VideoCapture(path)

  #initialize frame number to 0
  f = 0

  #to store the frames of the video
  while(capture.isOpened()):
    ret, frame = capture.read()
    if ret == False:
      break

    # to save the frame to a specified file
    filename = os.path.join(outFolder, f"frame_{f}.jpg")
    cv2.imwrite(filename, frame)
    f += 1

  capture.release()

print("Frames extracted.")

# match labels from xml file to video frames

data = []
filePath = "/content/videos/captures"

for item in xmlFiles:

  if item["outside"] == 1:
    continue

  video = item["file"]
  frame = item["frame"]

  path1 = os.path.join(filePath, video, f"frame_{frame}.jpg")
  if not os.path.exists(path1):
    continue

  data.append({
      "frame" : path1,
      "label": item["label"],
      "boundbox": item["boundbox"]
  })

print("Dataset size:", len(data))

# convert to pytorch tensors (in baseball class - done above)

#final framing format should be (frame - video image, target - bounding box)

0 baseball 25 0 2100.10 1649.50 2160.00 1740.60 1
0 baseball 26 0 1512.00 1763.00 1734.00 1875.70 1
0 baseball 27 0 1730.60 1880.10 1931.34 1965.65 1
0 baseball 28 1 1730.60 1880.10 1931.34 1965.65 1


In [ ]:
# label = root[2].attrib["label"]  #might need to change this   #are all labels and boxes within [2]?
# for child in root[2]:            
#     print(child.attrib)
#     print(child[0].attrib)

for child in root[2]:               #getting label values from binding boxes
    frame = child.attrib["frame"]
    outside = child.attrib["outside"]
    xtl = child.attrib["xtl"]
    ytl = child.attrib["ytl"]
    xbr = child.attrib["xbr"]
    ybr = child.attrib["ybr"]
    if root[2][0][0].text == "true":   #if statement to change moving to 1 and 0
        moving = 1
    else:
        moving = 0
    print(id, label, frame, outside, xtl, ytl, xbr, ybr, moving)


In [156]:
for track in root.findall("track"):
    for child in track:
        for tag in child:     
            if tag.text == "true":  
                 moving = 1
            else:
                 moving = 0
            print(moving)


1
1
1
1


In [ ]:
#progress on for loop for videos

import cv2
import os

videoFolder = "/content/videos/Movies"
frameFolder = "/content/videos/captures"

#creating folder (for google collab)
# output_dir = "videos/Captures"   #folder now created
os.makedirs(output_dir, exist_ok=True)

#for loop to go through videos
for video_file in os.listdir(videoFolder)
#capture video
capture = cv2.VideoCapture("/content/videos/Movies/IMG_0030.mov")
print("Opened:", capture.isOpened())
#initialize frame number to 0
f = 0
#to store the frames of the video
while(capture.isOpened()):
  ret, frame = capture.read()
  if ret == False:
    break

  # to save the frame to a specified file
  filename = os.path.join(output_dir, f"Frame_{f}.jpg")
  cv2.imwrite(filename, frame)
  f += 1

capture.release()
print("Done saving frames!")

In [18]:
import xml.etree.ElementTree as ET
import os
import pandas as pd

folder = "C:/Users/andre/Downloads/baseballannotated/Annotations"
xmlFiles = []
for file in os.listdir(folder):

    filePath = os.path.join(folder, file)
    tree = ET.parse(filePath)
    root = tree.getroot()
  
    for track in root.findall("track"):  #need a for loop for each id within xml file
        id = track.attrib["id"]
        label = track.attrib["label"]           #baseball label

        for child in track.findall("box"):               #getting label values from binding boxes
            frame = child.attrib["frame"]
            outside = child.attrib["outside"]
            xtl = child.attrib["xtl"]
            ytl = child.attrib["ytl"]
            xbr = child.attrib["xbr"]
            ybr = child.attrib["ybr"]

            for tag in child:     
                if tag.text == "true":  
                    moving = 1
                else:
                    moving = 0
            
            print(file, id, label, frame, outside, xtl, ytl, xbr, ybr, moving)
            xmlFiles.append([file, id, label, frame, outside, xtl, ytl, xbr, ybr, moving])

# print(xmlFiles[0])

dusty_1.xml 0 baseball 5 0 1058.12 518.47 1077.95 537.34 0
dusty_1.xml 0 baseball 6 0 1059.88 518.63 1079.71 537.50 0
dusty_1.xml 0 baseball 7 0 1061.64 518.79 1081.47 537.66 0
dusty_1.xml 0 baseball 8 0 1063.24 519.22 1083.07 538.09 0
dusty_1.xml 0 baseball 9 0 1064.84 519.64 1084.67 538.51 0
dusty_1.xml 0 baseball 10 0 1066.44 520.07 1086.27 538.94 0
dusty_1.xml 0 baseball 11 0 1068.20 520.07 1088.02 538.94 0
dusty_1.xml 0 baseball 12 0 1069.96 520.07 1089.78 538.94 0
dusty_1.xml 0 baseball 13 0 1070.49 520.07 1090.32 538.94 0
dusty_1.xml 0 baseball 14 0 1071.02 520.07 1090.85 538.94 0
dusty_1.xml 0 baseball 15 0 1071.55 520.07 1091.38 538.94 0
dusty_1.xml 0 baseball 16 0 1072.83 520.07 1092.66 538.94 0
dusty_1.xml 0 baseball 17 0 1074.11 520.07 1093.94 538.94 0
dusty_1.xml 0 baseball 18 0 1074.65 519.54 1094.47 538.41 0
dusty_1.xml 0 baseball 19 0 1075.18 519.00 1095.01 537.87 0
dusty_1.xml 0 baseball 20 0 1075.71 518.47 1095.54 537.34 0
dusty_1.xml 0 baseball 21 0 1076.35 518.47 10